# How to Evaluate AI Agents - A Head-to-Head Framework Comparison

## Strands Agents vs PydanticAI vs DeepEval

**The question:** You built an AI agent. It calls tools, reasons over data, and produces answers. How do you know if it's good? Three open-source frameworks offer different approaches to this problem. This notebook runs the **exact same evaluation tasks** on the **exact same agent outputs** across all three, so you can see the real differences.

## Why these 3 frameworks (and not CrewAI, LangGraph, or AutoGen)?

We compared 8 agent frameworks in our [research](../FRAMEWORK_COMPARISON.md). Most of them (CrewAI, LangGraph, AutoGen, OpenAI Agents SDK, Google ADK) are frameworks for **building** agents. They do not include evaluation-specific libraries.

These 3 were selected because they are the only ones with **dedicated evaluation SDKs**:

| Framework | Evaluation Library | What It Provides |
|-----------|-------------------|-----------------|
| **Strands Agents** | `strands-agents-evals` | OutputEvaluator, TrajectoryEvaluator, ToolCalled, ActorSimulator, Experiment runner |
| **PydanticAI** | `pydantic-evals` | LLMJudge, Dataset with YAML serialization, report diffing, HasMatchingSpan |
| **DeepEval** | `deepeval` (standalone) | 30+ built-in metrics: GEval, HallucinationMetric, FaithfulnessMetric, ToolCorrectnessMetric |

**What about the others?**

| Framework | Why Not Included |
|-----------|-----------------|
| **CrewAI** | `crewai test` exists but only supports OpenAI and provides basic 1-10 scoring. No rubrics, no trajectory eval, no hallucination detection. |
| **LangGraph** | Evaluation lives in **LangSmith** (paid SaaS), not in the open-source framework. We compare open-source only. |
| **AutoGen** | Has AutoGen Bench for benchmarking, but no evaluation SDK with metrics comparable to these 3. |
| **OpenAI Agents SDK** | Provides tracing hooks but no evaluation library. You need to pair it with DeepEval or another tool. |
| **Google ADK** | Has `adk eval` CLI but it is tightly coupled to the Gemini ecosystem. |

**Bottom line:** If you use CrewAI, LangGraph, or AutoGen to build your agent, you still need one of these 3 evaluation frameworks (or a combination) to evaluate it. DeepEval in particular is framework-agnostic and works with any agent.

---

## What we evaluate (and why)

| Round | Evaluation Task | Why It Matters |
|:-----:|----------------|----------------|
| 1 | **Output Quality** (LLM-as-Judge) | Is the agent's answer helpful and accurate? |
| 2 | **Hallucination Detection** | Did the agent fabricate information not in the source context? |
| 3 | **Tool Correctness** | Did the agent call the right tools? |

**How the comparison works:**

All three frameworks receive the **same test data** (3 pre-computed agent responses) and use the **same judge model** (GPT-4o-mini via OpenAI). The only variable is the framework API. This isolates the framework differences from model differences.

**The three frameworks side by side:**

| Framework | Approach | How It Evaluates |
|-----------|----------|-----------------|
| **Strands Agents** (`strands-agents-evals`) | Agent-native | `OutputEvaluator` with a rubric you write. Part of the agent runtime. |
| **PydanticAI** (`pydantic-evals`) | Type-safe datasets | `LLMJudge` with typed datasets. Separate from the agent framework. |
| **DeepEval** | Standalone library | 30+ built-in metrics (`GEval`, `HallucinationMetric`). Framework-agnostic. |

In [ ]:
# %pip install strands-agents strands-agents-evals strands-agents-tools pydantic-evals pydantic-ai deepeval boto3 tabulate nest-asyncio

## Setup: Shared Test Data

For a fair comparison, all three frameworks evaluate the **exact same data**. We define 3 pre-computed agent responses instead of calling a live agent. This way, differences in scores are caused by the evaluation framework, not by randomness in agent behavior.

**The 3 test cases:**

| Name | What the agent said | Quality |
|------|-------------------|---------|
| `flight_search_good` | Lists 3 specific flights with airlines, times, prices | Good |
| `flight_search_hallucinated` | Lists flights but adds fabricated awards and a fake airline | Bad (hallucinated) |
| `weather_query` | Reports weather with accurate details plus a sightseeing suggestion | Mostly good |

We also define two **shared rubrics** (scoring criteria) that all frameworks use:
- **Quality rubric**: "Rate helpfulness 0-1. Score 0.8+ for specific details, 0.4-0.7 for partial, below 0.4 for vague."
- **Hallucination rubric**: "Score 1.0 if grounded in context. Score 0.0 if fabricated details present."

In [ ]:
import time
import nest_asyncio

# Fix for Jupyter: PydanticAI's evaluate_sync() uses asyncio.run_until_complete()
# which fails in notebooks because Jupyter already has a running event loop.
# nest_asyncio patches this so nested event loops work.
nest_asyncio.apply()

# All three frameworks use the same judge model for a fair comparison
JUDGE_MODEL = "gpt-4o-mini"

# --- 3 Pre-Computed Agent Responses ---
# We use pre-computed outputs instead of a live agent so the comparison
# isolates framework differences, not agent randomness.

TEST_CASES = [
    {
        "name": "flight_search_good",
        "input": "Find flights from NYC to London for next Friday",
        "agent_output": (
            "I found 3 flights for next Friday:\n"
            "1. British Airways BA117 - Departs JFK 7:00 PM, arrives LHR 7:00 AM - $450\n"
            "2. Delta DL1 - Departs JFK 9:30 PM, arrives LHR 9:30 AM - $520\n"
            "3. United UA100 - Departs EWR 8:15 PM, arrives LHR 8:15 AM - $480\n"
            "All flights are nonstop and approximately 7 hours."
        ),
        "expected_output": "Should list specific flights with airline names, flight numbers, departure times, and prices",
        "context": [
            "BA117 departs JFK at 7PM, arrives LHR 7AM, costs $450.",
            "DL1 departs JFK at 9:30PM, arrives LHR 9:30AM, costs $520.",
            "UA100 departs EWR at 8:15PM, arrives LHR 8:15AM, costs $480.",
        ],
        "tools_called": ["search_flights", "check_availability"],
        "expected_tools": ["search_flights"],
    },
    {
        "name": "flight_search_hallucinated",
        "input": "Find flights from NYC to London for next Friday",
        "agent_output": (
            "I found several options:\n"
            "1. British Airways BA117 - $450 (Award-winning service with complimentary champagne)\n"
            "2. Delta DL1 - $520 (Recently rated #1 transatlantic airline by TripAdvisor)\n"
            "3. Virgin Atlantic VS10 - $399 (Cheapest option with free lounge access)\n"
            "I recommend Virgin Atlantic for the best value."
        ),
        "expected_output": "Should list specific flights with accurate details from context only",
        "context": [
            "BA117 departs JFK at 7PM, arrives LHR 7AM, costs $450.",
            "DL1 departs JFK at 9:30PM, arrives LHR 9:30AM, costs $520.",
        ],
        "tools_called": ["search_flights"],
        "expected_tools": ["search_flights"],
    },
    {
        "name": "weather_query",
        "input": "What's the weather in Paris tomorrow?",
        "agent_output": (
            "Tomorrow in Paris: 18C (64F), partly cloudy with a 20% chance of rain "
            "in the afternoon. Wind from the west at 15 km/h. "
            "Great weather for sightseeing!"
        ),
        "expected_output": "Temperature, conditions, and precipitation chance",
        "context": [
            "Paris forecast: 18C, partly cloudy, 20% rain chance afternoon, west wind 15 km/h.",
        ],
        "tools_called": ["get_weather"],
        "expected_tools": ["get_weather"],
    },
]

# --- Shared Rubrics ---
# Both Strands and PydanticAI use these rubrics. DeepEval uses them as "criteria" in GEval.

QUALITY_RUBRIC = (
    "Rate the response on helpfulness (0 to 1). A helpful response includes "
    "specific, actionable information directly answering the question. "
    "Penalize vague, generic, or off-topic responses. "
    "Score 0.8+ for responses with specific details (names, numbers, dates). "
    "Score 0.4-0.7 for partially helpful responses. "
    "Score below 0.4 for unhelpful or irrelevant responses."
)

HALLUCINATION_RUBRIC = (
    "Score 1.0 if the response ONLY contains information present in the provided context. "
    "Score 0.0 if the response includes ANY fabricated details such as awards, rankings, "
    "services, or facts not mentioned in the context. "
    "Score 0.3-0.7 for responses that are mostly grounded but include minor embellishments."
)

print(f"Loaded {len(TEST_CASES)} test cases: {[tc['name'] for tc in TEST_CASES]}")
print(f"Judge model: {JUDGE_MODEL}")
print(f"nest_asyncio applied (fixes Jupyter event loop conflict)")

---

## Round 1: Output Quality (LLM-as-Judge)

**The task:** An LLM judge reads the agent's response and scores it 0-1 based on a rubric (scoring criteria). This is the most fundamental evaluation technique.

**How each framework does it:**

```
Strands:    Case(input, expected_output) → OutputEvaluator(rubric) → Experiment → score
PydanticAI: Case(inputs, expected_output) → LLMJudge(rubric) → Dataset → score + pass/fail
DeepEval:   LLMTestCase(input, actual_output) → GEval(criteria) → evaluate() → score
```

Notice the API differences: Strands uses `Experiment` + `Case`, PydanticAI uses `Dataset` + `Case`, DeepEval uses `evaluate()` + `LLMTestCase`. Same concept, different vocabulary.

### Strands Agents

**Key pattern:** You create `Case` objects, an `OutputEvaluator` with your rubric, wrap them in an `Experiment`, and call `run_evaluations(task_fn)`. The `task_fn` receives a Case and returns the agent output (here we return pre-computed responses).

**What to look for in the results:** A Rich table with scores and reasons for each test case.

In [ ]:
from strands_evals import Experiment, Case
from strands_evals.evaluators import OutputEvaluator

# Step 1: Wrap each test case in a Strands Case object
# The Case holds the question (input) and what a good answer looks like (expected_output)
strands_cases = [
    Case(name=tc["name"], input=tc["input"], expected_output=tc["expected_output"])
    for tc in TEST_CASES
]

# Step 2: Create the judge with a rubric (scoring criteria)
# The rubric tells the LLM HOW to score. Without it, scores are inconsistent.
strands_quality_eval = OutputEvaluator(rubric=QUALITY_RUBRIC, model=JUDGE_MODEL)

# Step 3: Define a task function that returns the agent's output for each case
# In production, this would call agent(case.input). Here we use pre-computed outputs.
agent_outputs = {tc["name"]: tc["agent_output"] for tc in TEST_CASES}

def strands_task(case):
    return agent_outputs[case.name]

# Step 4: Run the evaluation
# Experiment combines cases + evaluators and runs all combinations
start = time.time()
strands_experiment = Experiment(cases=strands_cases, evaluators=[strands_quality_eval])
strands_quality_reports = strands_experiment.run_evaluations(strands_task)
strands_quality_time = time.time() - start

print(f"Strands: Output Quality ({strands_quality_time:.1f}s)")
strands_quality_reports[0].display()

### PydanticAI

**Key difference from Strands:** PydanticAI wraps cases in a `Dataset` (not `Experiment`). The evaluator is `LLMJudge` (not `OutputEvaluator`). PydanticAI also splits the result into two outputs: a **score** (0-1 number) and an **assertion** (pass/fail boolean). You can enable one or both.

**What to look for:** The `quality_score` column (numeric) and `quality_pass` column (True/False). Strands only gives a score.

In [ ]:
"""Round 1B: Output Quality with PydanticAI."""

from pydantic_evals import Case as PydanticCase, Dataset
from pydantic_evals.evaluators import LLMJudge

# Build PydanticAI dataset
pydantic_cases = [
    PydanticCase(
        name=tc["name"],
        inputs=tc["input"],
        expected_output=tc["expected_output"],
    )
    for tc in TEST_CASES
]

pydantic_dataset = Dataset(
    name="output_quality",
    cases=pydantic_cases,
    evaluators=[
        LLMJudge(
            rubric=QUALITY_RUBRIC,
            model="openai:gpt-4o-mini",
            include_input=True,
            include_expected_output=True,
            score={"include_reason": True, "evaluation_name": "quality_score"},
            assertion={"include_reason": True, "evaluation_name": "quality_pass"},
        ),
    ],
)

def pydantic_task(inputs: str) -> str:
    """Return pre-computed output for fair comparison."""
    for tc in TEST_CASES:
        if tc["input"] == inputs:
            return tc["agent_output"]
    return "No output available"

# Run evaluation and measure time
start = time.time()
pydantic_quality_report = pydantic_dataset.evaluate_sync(pydantic_task)
pydantic_quality_time = time.time() - start

print(f"\n--- PydanticAI: Output Quality ({pydantic_quality_time:.1f}s) ---")
pydantic_quality_report.print(include_input=True, include_averages=True)

### DeepEval

**Key difference from Strands and PydanticAI:** DeepEval does not use rubrics directly. Instead, it uses `GEval` which takes `criteria` (a plain text description) and `evaluation_params` (which fields to pass to the judge). DeepEval also uses `LLMTestCase` instead of `Case`, and `evaluate()` instead of `Experiment`.

**Another difference:** DeepEval defaults to OpenAI natively (just set `OPENAI_API_KEY`). Strands and PydanticAI also support OpenAI but through their own model provider abstraction.

**What to look for:** Per-test-case scores printed in the console. DeepEval does not produce a Rich table like Strands and PydanticAI.

In [ ]:
from deepeval import evaluate
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.evaluate.configs import AsyncConfig, DisplayConfig

# Step 1: Wrap test cases in DeepEval's LLMTestCase format
# Note: DeepEval uses "actual_output" (not "expected_output" for the agent's response)
deepeval_quality_cases = [
    LLMTestCase(
        name=tc["name"],
        input=tc["input"],
        actual_output=tc["agent_output"],        # What the agent said
        expected_output=tc["expected_output"],    # What a good answer looks like
    )
    for tc in TEST_CASES
]

# Step 2: Create the GEval metric
# GEval takes "criteria" (like a rubric) and "evaluation_params" (which fields the judge sees)
helpfulness_metric = GEval(
    name="Helpfulness",
    criteria=QUALITY_RUBRIC,
    evaluation_params=[
        LLMTestCaseParams.INPUT,           # The question
        LLMTestCaseParams.ACTUAL_OUTPUT,   # The agent's response
        LLMTestCaseParams.EXPECTED_OUTPUT, # What a good answer looks like
    ],
    threshold=0.5,  # Score >= 0.5 = pass
)

# Step 3: Run evaluation
# DeepEval uses evaluate() (not Experiment). It defaults to OpenAI when OPENAI_API_KEY is set.
start = time.time()
deepeval_quality_result = evaluate(
    test_cases=deepeval_quality_cases,
    metrics=[helpfulness_metric],
    async_config=AsyncConfig(run_async=False),
    display_config=DisplayConfig(print_results=True),
)
deepeval_quality_time = time.time() - start

print(f"\nDeepEval: Output Quality ({deepeval_quality_time:.1f}s)")
for tr in deepeval_quality_result.test_results:
    for md in tr.metrics_data or []:
        print(f"  {tr.name}: score={md.score:.2f}, pass={md.success}")

### Round 1: What did we learn?

| Aspect | Strands | PydanticAI | DeepEval |
|--------|---------|------------|---------|
| **Core class** | `OutputEvaluator` | `LLMJudge` | `GEval` |
| **Test case class** | `Case` | `Case` | `LLMTestCase` |
| **Runner** | `Experiment.run_evaluations()` | `Dataset.evaluate_sync()` | `evaluate()` |
| **Output format** | Score (0-1) + reason | Score + pass/fail + reason | Score (0-1) + reason |
| **Display** | Rich table (`.display()`) | Rich table (`.print()`) | Console text |

**Key insight:** All three produce scores. The difference is ergonomics. Strands is the most concise (4 lines of setup). PydanticAI gives you score + pass/fail separately. DeepEval has more configuration options (evaluation_params, threshold).

---

## Round 2: Hallucination Detection

**The task:** Given a response and its source context, does the response contain fabricated information?

This is where the frameworks diverge most. Strands and PydanticAI use the same general-purpose judge with a hallucination rubric. DeepEval has a **dedicated `HallucinationMetric`** that decomposes the response into individual claims and checks each against the context.

```
Strands:    OutputEvaluator(rubric="check if grounded...") → single score
PydanticAI: LLMJudge(rubric="check if grounded...") → score + pass/fail
DeepEval:   HallucinationMetric() → decompose claims → check each → score
```

**What to watch for:** The `flight_search_hallucinated` case fabricates awards, champagne service, and a fake airline (Virgin Atlantic VS10). Can each framework catch it?

### Strands Agents

**Challenge:** Strands `OutputEvaluator` does not have a dedicated `context` field. We work around this by putting the context in the `expected_output` field and telling the rubric to compare against it.

In [ ]:
"""Round 2A: Hallucination Detection with Strands Agents.

Strands uses a general-purpose OutputEvaluator with a hallucination-focused rubric.
The context is passed via expected_output so the judge can compare against it.
"""

strands_hallucination_cases = [
    Case(
        name=tc["name"],
        input=tc["input"] + "\n\nContext:\n" + "\n".join(tc["context"]),
        expected_output="\n".join(tc["context"]),  # Ground truth context
    )
    for tc in TEST_CASES
]

strands_hallucination_eval = OutputEvaluator(
    rubric=HALLUCINATION_RUBRIC,
    model=JUDGE_MODEL,
)

def strands_hallucination_task(case):
    """Return pre-computed output."""
    for tc in TEST_CASES:
        if tc["name"] == case.name:
            return tc["agent_output"]
    return ""

start = time.time()
strands_hallucination_exp = Experiment(
    cases=strands_hallucination_cases, evaluators=[strands_hallucination_eval]
)
strands_hallucination_reports = strands_hallucination_exp.run_evaluations(strands_hallucination_task)
strands_hallucination_time = time.time() - start

print(f"\n--- Strands: Hallucination Detection ({strands_hallucination_time:.1f}s) ---")
strands_hallucination_reports[0].display()

### PydanticAI

**Key difference:** PydanticAI uses typed inputs. We define a `HallucinationInput` TypedDict with `query` + `context` fields. This is more structured than Strands' string concatenation workaround, but requires more boilerplate.

In [ ]:
"""Round 2B: Hallucination Detection with PydanticAI.

PydanticAI uses LLMJudge with include_expected_output=True so the judge
can compare the response against the ground truth context.
"""

from typing_extensions import TypedDict


class HallucinationInput(TypedDict):
    query: str
    context: str


pydantic_hallucination_cases = [
    PydanticCase(
        name=tc["name"],
        inputs=HallucinationInput(
            query=tc["input"],
            context="\n".join(tc["context"]),
        ),
        expected_output="\n".join(tc["context"]),
    )
    for tc in TEST_CASES
]

pydantic_hallucination_dataset = Dataset(
    name="hallucination_detection",
    cases=pydantic_hallucination_cases,
    evaluators=[
        LLMJudge(
            rubric=HALLUCINATION_RUBRIC,
            model="openai:gpt-4o-mini",
            include_input=True,
            include_expected_output=True,
            score={"include_reason": True, "evaluation_name": "groundedness"},
            assertion={"include_reason": True, "evaluation_name": "no_hallucination"},
        ),
    ],
)

def pydantic_hallucination_task(inputs: HallucinationInput) -> str:
    """Return pre-computed output."""
    for tc in TEST_CASES:
        if tc["input"] == inputs["query"]:
            return tc["agent_output"]
    return ""

start = time.time()
pydantic_hallucination_report = pydantic_hallucination_dataset.evaluate_sync(
    pydantic_hallucination_task
)
pydantic_hallucination_time = time.time() - start

print(f"\n--- PydanticAI: Hallucination Detection ({pydantic_hallucination_time:.1f}s) ---")
pydantic_hallucination_report.print(include_input=True, include_averages=True, include_reasons=True)

### DeepEval (Dedicated HallucinationMetric)

**Key difference:** DeepEval has a **dedicated `context` field** in `LLMTestCase`. No workarounds needed. And `HallucinationMetric` does not need a rubric at all. It automatically decomposes the response into claims and checks each against the context.

This is the most specialized approach. The tradeoff: you cannot customize the evaluation criteria (no rubric).

In [ ]:
"""Round 2C: Hallucination Detection with DeepEval.

DeepEval has a purpose-built HallucinationMetric that decomposes claims
and checks each against the provided context -- more granular than rubric-based.
"""

from deepeval.metrics import HallucinationMetric

deepeval_hallucination_cases = [
    LLMTestCase(
        name=tc["name"],
        input=tc["input"],
        actual_output=tc["agent_output"],
        context=tc["context"],  # DeepEval has a dedicated context field
    )
    for tc in TEST_CASES
]

deepeval_hallucination_metric = HallucinationMetric(
    threshold=0.5,
    # model defaults to OpenAI when OPENAI_API_KEY is set
)

start = time.time()
deepeval_hallucination_result = evaluate(
    test_cases=deepeval_hallucination_cases,
    metrics=[deepeval_hallucination_metric],
    async_config=AsyncConfig(run_async=False),
    display_config=DisplayConfig(print_results=True),
)
deepeval_hallucination_time = time.time() - start

print(f"\n--- DeepEval: Hallucination Detection ({deepeval_hallucination_time:.1f}s) ---")
for tr in deepeval_hallucination_result.test_results:
    for md in tr.metrics_data or []:
        print(f"  {tr.name}: score={md.score:.2f}, pass={md.success}")
        if md.reason:
            print(f"    reason: {md.reason[:150]}...")

---

## Round 3: Tool Correctness

**The task:** Did the agent call the right tools? Our test data includes `tools_called` (what the agent did) and `expected_tools` (what it should have done).

**Why this matters:** A wrong tool call can trigger real-world side effects (booking the wrong hotel, calling the wrong API). Output quality can be perfect while tool selection is wrong.

**The framework split:**

| Framework | Tool Evaluation Support |
|-----------|----------------------|
| **Strands** | `ToolCalled` (deterministic, free) + `TrajectoryEvaluator` (LLM-based) |
| **PydanticAI** | `HasMatchingSpan` (requires OpenTelemetry traces) |
| **DeepEval** | `ToolCorrectnessMetric` (dedicated, compares ToolCall objects) |

### Strands Agents (Two Approaches)

**Approach 1: `ToolCalled`** — A deterministic check: "Was `search_flights` called?" Binary yes/no. No LLM needed, instant, free.

**Approach 2: `TrajectoryEvaluator`** — An LLM judge evaluates the full tool sequence against a rubric. Can assess whether the tool order was logical and whether unnecessary calls were made.

In [ ]:
"""Round 3A: Tool Correctness with Strands Agents.

Strands offers both deterministic checks (ToolCalled -- zero LLM cost)
and LLM-based trajectory evaluation (TrajectoryEvaluator).
"""

from strands_evals.evaluators import TrajectoryEvaluator, ToolCalled

strands_tool_cases = [
    Case(
        name=tc["name"],
        input=tc["input"],
        expected_trajectory=tc["expected_tools"],
    )
    for tc in TEST_CASES
]

# Approach 1: Deterministic check (zero cost, instant)
strands_deterministic_eval = ToolCalled(tool_name="search_flights")

# Approach 2: LLM-based trajectory evaluation
strands_trajectory_eval = TrajectoryEvaluator(
    rubric=(
        "Evaluate whether the agent called the correct tools in a logical order. "
        "The agent should use search tools before booking tools. "
        "Unnecessary tool calls should be penalized."
    ),
    model=JUDGE_MODEL,
)

def strands_tool_task(case):
    """Return pre-computed output with trajectory."""
    for tc in TEST_CASES:
        if tc["name"] == case.name:
            trajectory = [{"name": t} for t in tc["tools_called"]]
            return {"output": tc["agent_output"], "trajectory": trajectory}
    return {"output": "", "trajectory": []}

# Run deterministic evaluation (instant, zero cost)
start = time.time()
strands_det_exp = Experiment(cases=strands_tool_cases, evaluators=[strands_deterministic_eval])
strands_det_reports = strands_det_exp.run_evaluations(strands_tool_task)
strands_det_time = time.time() - start

print(f"--- Strands Deterministic: ToolCalled ({strands_det_time:.2f}s, $0 cost) ---")
strands_det_reports[0].display()

# Run LLM-based trajectory evaluation
start = time.time()
strands_traj_exp = Experiment(cases=strands_tool_cases, evaluators=[strands_trajectory_eval])
strands_traj_reports = strands_traj_exp.run_evaluations(strands_tool_task)
strands_traj_time = time.time() - start

print(f"\n--- Strands LLM-based: TrajectoryEvaluator ({strands_traj_time:.1f}s) ---")
strands_traj_reports[0].display()

### DeepEval (ToolCorrectnessMetric)

**Key difference:** DeepEval uses structured `ToolCall` objects with `name` and optional `input_parameters`. You pass `tools_called` (what the agent did) and `expected_tools` (ground truth). The metric compares them.

**Options:** `should_consider_ordering=True` penalizes wrong order. `should_exact_match=True` requires exact parameter match.

In [ ]:
"""Round 3B: Tool Correctness with DeepEval.

DeepEval has a dedicated ToolCorrectnessMetric with structured ToolCall objects.
It supports ordering validation and exact match options.
"""

from deepeval.metrics import ToolCorrectnessMetric
from deepeval.test_case import ToolCall

deepeval_tool_cases = [
    LLMTestCase(
        name=tc["name"],
        input=tc["input"],
        actual_output=tc["agent_output"],
        tools_called=[ToolCall(name=t) for t in tc["tools_called"]],
        expected_tools=[ToolCall(name=t) for t in tc["expected_tools"]],
    )
    for tc in TEST_CASES
]

deepeval_tool_metric = ToolCorrectnessMetric(
    threshold=0.5,
    # model defaults to OpenAI when OPENAI_API_KEY is set
    should_consider_ordering=False,
    should_exact_match=False,
)

start = time.time()
deepeval_tool_result = evaluate(
    test_cases=deepeval_tool_cases,
    metrics=[deepeval_tool_metric],
    async_config=AsyncConfig(run_async=False),
    display_config=DisplayConfig(print_results=True),
)
deepeval_tool_time = time.time() - start

print(f"\n--- DeepEval: ToolCorrectness ({deepeval_tool_time:.1f}s) ---")
for tr in deepeval_tool_result.test_results:
    for md in tr.metrics_data or []:
        print(f"  {tr.name}: score={md.score:.2f}, pass={md.success}")

---

## Final Summary

We ran the same 3 evaluation tasks across all 3 frameworks. The cells below compile the results into comparison tables so you can see the differences at a glance.

In [ ]:
"""Final Summary: Compile all results into comparison tables."""

from tabulate import tabulate

# --- Timing Comparison ---
timing_data = [
    ["Output Quality (LLM-as-Judge)", f"{strands_quality_time:.1f}s", f"{pydantic_quality_time:.1f}s", f"{deepeval_quality_time:.1f}s"],
    ["Hallucination Detection", f"{strands_hallucination_time:.1f}s", f"{pydantic_hallucination_time:.1f}s", f"{deepeval_hallucination_time:.1f}s"],
    ["Tool Correctness (Deterministic)", f"{strands_det_time:.2f}s", "N/A", "N/A"],
    ["Tool Correctness (LLM-based)", f"{strands_traj_time:.1f}s", "N/A", f"{deepeval_tool_time:.1f}s"],
]

print("=" * 80)
print("TIMING COMPARISON (3 test cases each)")
print("=" * 80)
print(tabulate(timing_data, headers=["Evaluation", "Strands", "PydanticAI", "DeepEval"], tablefmt="grid"))

# --- Feature Comparison ---
feature_data = [
    ["LLM-as-Judge", "OutputEvaluator", "LLMJudge", "GEval"],
    ["Hallucination (dedicated)", "No (via rubric)", "No (via rubric)", "HallucinationMetric"],
    ["Tool correctness (dedicated)", "ToolCalled (deterministic)", "HasMatchingSpan", "ToolCorrectnessMetric"],
    ["Trajectory evaluation", "TrajectoryEvaluator", "Custom (SpanTree)", "N/A"],
    ["Faithfulness (dedicated)", "FaithfulnessEvaluator*", "No (via rubric)", "FaithfulnessMetric"],
    ["Multi-agent eval", "InteractionsEvaluator", "N/A", "N/A"],
    ["Multi-turn simulation", "ActorSimulator", "N/A", "ConversationalTestCase"],
    ["Deterministic checks", "Equals, Contains, ToolCalled", "Equals, Contains, IsInstance", "N/A"],
    ["Bedrock native", "Yes", "Yes (anthropic:)", "Custom wrapper"],
    ["Report diffing", "No", "Yes (baseline=)", "Via Confident AI"],
    ["Dataset serialization", "JSON", "YAML + JSON", "JSON + CSV"],
    ["Total built-in metrics", "12", "6 + custom", "30+"],
]

print("\n")
print("=" * 80)
print("FEATURE COMPARISON")
print("=" * 80)
print(tabulate(feature_data, headers=["Feature", "Strands", "PydanticAI", "DeepEval"], tablefmt="grid"))
print("\n* FaithfulnessEvaluator requires OpenTelemetry trace-based setup")

In [ ]:
"""Visual comparison charts."""

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = 'white'

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Colors for each framework
colors = {'Strands': '#2196F3', 'PydanticAI': '#4CAF50', 'DeepEval': '#FF9800'}

# --- Chart 1: Execution Time by Round ---
ax = axes[0]
rounds = ['Output\nQuality', 'Hallucination\nDetection', 'Tool\nCorrectness']
strands_times = [strands_quality_time, strands_hallucination_time, strands_traj_time]
pydantic_times = [pydantic_quality_time, pydantic_hallucination_time, 0]
deepeval_times = [deepeval_quality_time, deepeval_hallucination_time, deepeval_tool_time]

x = range(len(rounds))
width = 0.25
ax.bar([i - width for i in x], strands_times, width, label='Strands', color=colors['Strands'])
ax.bar(x, pydantic_times, width, label='PydanticAI', color=colors['PydanticAI'])
ax.bar([i + width for i in x], deepeval_times, width, label='DeepEval', color=colors['DeepEval'])
ax.set_ylabel('Time (seconds)')
ax.set_title('Execution Time by Round')
ax.set_xticks(x)
ax.set_xticklabels(rounds, fontsize=9)
ax.legend(fontsize=8)

# --- Chart 2: Feature Coverage (what each framework supports) ---
ax = axes[1]
features = ['LLM Judge', 'Hallucination\n(dedicated)', 'Tool\nCorrectness', 'Trajectory\nEval', 'Deterministic\nChecks', 'Multi-agent']
strands_feat = [1, 0, 1, 1, 1, 1]   # 1 = has it, 0 = doesn't
pydantic_feat = [1, 0, 1, 0, 1, 0]
deepeval_feat = [1, 1, 1, 0, 0, 0]

x = range(len(features))
ax.barh([f + 0.2 for f in x], strands_feat, 0.2, label='Strands', color=colors['Strands'], alpha=0.8)
ax.barh(x, pydantic_feat, 0.2, label='PydanticAI', color=colors['PydanticAI'], alpha=0.8)
ax.barh([f - 0.2 for f in x], deepeval_feat, 0.2, label='DeepEval', color=colors['DeepEval'], alpha=0.8)
ax.set_yticks(x)
ax.set_yticklabels(features, fontsize=8)
ax.set_xlabel('Has Feature (1 = Yes)')
ax.set_title('Feature Coverage')
ax.legend(fontsize=8, loc='lower right')
ax.set_xlim(0, 1.3)

# --- Chart 3: Setup Complexity (lines of code per round) ---
ax = axes[2]
categories = ['Output Quality\n(Round 1)', 'Hallucination\n(Round 2)', 'Tool Correctness\n(Round 3)']
strands_loc = [8, 10, 15]
pydantic_loc = [15, 20, 0]  # 0 = not applicable
deepeval_loc = [12, 8, 12]

x = range(len(categories))
ax.bar([i - width for i in x], strands_loc, width, label='Strands', color=colors['Strands'])
ax.bar(x, pydantic_loc, width, label='PydanticAI', color=colors['PydanticAI'])
ax.bar([i + width for i in x], deepeval_loc, width, label='DeepEval', color=colors['DeepEval'])
ax.set_ylabel('Lines of Code')
ax.set_title('Setup Complexity')
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=9)
ax.legend(fontsize=8)

plt.tight_layout()
plt.suptitle('Framework Comparison: Strands vs PydanticAI vs DeepEval', fontsize=13, fontweight='bold', y=1.02)
plt.show()

## Conclusion

### When to Use Each

| Use Case | Best Choice | Why |
|----------|-------------|-----|
| You use **Strands Agents** for your agent | **Strands evals** | Same ecosystem, hooks, built-in metrics |
| You want **type-safe** evaluation pipelines | **PydanticAI** | Strongest typing, YAML datasets, report diffing |
| You need the **most metrics out-of-the-box** | **DeepEval** | 30+ metrics, hallucination/faithfulness specialized |
| You need **trajectory + tool** evaluation | **Strands** | Built-in extractors, TrajectoryEvaluator, deterministic checks |
| You need **multi-agent** evaluation | **Strands** | InteractionsEvaluator, Swarm/Graph orchestration |
| You want **framework-agnostic** evaluation | **DeepEval** | Works with any agent framework, no coupling |
| You want **report comparison** across runs | **PydanticAI** | Baseline diffing built into `.print()` |

### The Bottom Line

There is no single "best" framework. The evaluation concepts (LLM-as-judge, trajectory scoring, hallucination detection) are framework-independent. Choose based on your stack:

- **Already using Strands Agents** → Strands evals (tightest integration)
- **Want type safety and structured pipelines** → PydanticAI
- **Want maximum built-in metrics** → DeepEval

All three are production-quality. The best evaluation framework is the one you actually use.